In [1]:
# ============================================================
# WEEK 2 - DAY 4: Segment Profiling
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120

# ============================================================
# STEP 1: Load Everything & Attach Cluster Labels
# ============================================================

# Original customer features (unscaled — real values)
df_original = pd.read_csv('customer_features.csv')

# Cluster labels
df_clustered = pd.read_csv('clustered_customers.csv')

# Bulk buyers we removed (keep track of them)
df_bulk = pd.read_csv('bulk_buyers_removed.csv')

# Attach cluster labels to original data
# clustered_customers has same row order as customer_features
# (after bulk buyer removal)
df_original = df_original[~df_original['Customer ID'].isin(df_bulk['Customer ID'])]
df_original = df_original.reset_index(drop=True)
df_original['Cluster'] = df_clustered['Cluster']

print(f"Customers:        {len(df_original)}")
print(f"Bulk buyers removed: {len(df_bulk)}")
print(f"Clusters:         {df_original['Cluster'].nunique()}")
print(f"\nCluster distribution:")
print(df_original['Cluster'].value_counts().sort_index())

Customers:        4242
Bulk buyers removed: 43
Clusters:         5

Cluster distribution:
Cluster
0     570
1    1662
2    1800
3     200
4      10
Name: count, dtype: int64


In [2]:
# ============================================================
# STEP 2: Cluster Profiles — Core Statistics
# ============================================================
print("=" * 60)
print("STEP 2: Core Statistics per Cluster")
print("=" * 60)

# Key features to profile on (unscaled, real values)
PROFILE_COLS = [
    'NumOrders',
    'TotalSpent',
    'AvgOrderValue',
    'UniqueProducts',
    'CustomerLifetimeDays',
    'AvgDaysBetweenOrders',
    'Recency',
    'AvgQuantityPerOrder',
]

# Mean of each feature per cluster
profile = df_original.groupby('Cluster')[PROFILE_COLS].mean().round(2)
print("\nCluster Profiles (mean values):")
print(profile.to_string())

# Size and revenue share per cluster
cluster_summary = df_original.groupby('Cluster').agg(
    NumCustomers   = ('Customer ID', 'count'),
    TotalRevenue   = ('TotalSpent', 'sum'),
    AvgSpend       = ('TotalSpent', 'mean'),
    MedianSpend    = ('TotalSpent', 'median'),
    AvgRecency     = ('Recency', 'mean'),
    AvgOrders      = ('NumOrders', 'mean'),
).round(2)

cluster_summary['RevenueShare%'] = (
    cluster_summary['TotalRevenue'] / 
    cluster_summary['TotalRevenue'].sum() * 100
).round(2)

cluster_summary['CustomerShare%'] = (
    cluster_summary['NumCustomers'] / 
    cluster_summary['NumCustomers'].sum() * 100
).round(2)

print("\nCluster Business Summary:")
print(cluster_summary.to_string())

STEP 2: Core Statistics per Cluster

Cluster Profiles (mean values):
         NumOrders  TotalSpent  AvgOrderValue  UniqueProducts  CustomerLifetimeDays  AvgDaysBetweenOrders  Recency  AvgQuantityPerOrder
Cluster                                                                                                                                
0             3.54     2656.16         771.85          109.85                144.60                 36.51    77.05               489.12
1             5.14     1521.35         300.31           68.79                229.45                 54.80    43.60               167.91
2             1.42      310.79         234.63           20.61                 19.76                  8.28   147.22               134.29
3            21.50    10286.49         474.98          237.16                334.16                 18.83    16.04               269.08
4           115.90    77510.30         718.22          869.50                368.40                  3.37     3.70 